# Phase 5 — Rankings, Uncertainty and Robustness

This notebook reproduces Phase 4, creates the official rankings, runs stratified match-level bootstraps, and tests locked robustness scenarios. It uses the complete 2022–2026 period with no recency weighting.

In [1]:
from pathlib import Path
import hashlib
import json
import math

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import kendalltau, spearmanr

ROOT = Path.cwd()
INPUT_PATH = ROOT / 'phase3_outputs' / 'phase3_structural_pressure_labelled.csv'
PHASE4_DIR = ROOT / 'phase4_outputs'
OUTPUT_DIR = ROOT / 'phase5_outputs'
REPORT_PATH = ROOT / 'Phase 5 Rankings Uncertainty and Robustness Report.md'
MODEL_VERSION = 'ranking_uncertainty_v1'
MIN_SUPPORT = 100
FORMATS = ['IPL', 'T20I']
FIXED_REPLICATES = 5000
NESTED_REPLICATES = 1000
FIXED_SEED = 20261114
NESTED_SEED = 20261115
FIXED_BATCH = 100
NESTED_BATCH = 20
PREUPDATE_COMBINED_HASHES = {
    'Batter': 'd92f77fe035a0f0d65f3e24bbbc936d57244cc36bf2735a460afc55e93297c15',
    'Bowler': '6ef2c72567210965f68d3c8d84e0a0412ab31a10f889e78b1192be3f4f91a958',
}
PRIMARY_WEIGHTS = {
    'Batter': np.array([0.45, 0.25, 0.20, 0.10]),
    'Bowler': np.array([0.45, 0.30, 0.15, 0.10]),
}
WEIGHT_SCENARIOS = {
    'primary': PRIMARY_WEIGHTS,
    'balanced_quality': {
        'Batter': np.array([0.35, 0.35, 0.20, 0.10]),
        'Bowler': np.array([0.40, 0.40, 0.10, 0.10]),
    },
    'primary_outcome_heavy': {
        'Batter': np.array([0.60, 0.15, 0.15, 0.10]),
        'Bowler': np.array([0.60, 0.20, 0.15, 0.05]),
    },
    'gradient_heavy': {
        'Batter': np.array([0.35, 0.20, 0.30, 0.15]),
        'Bowler': np.array([0.35, 0.25, 0.25, 0.15]),
    },
}
THRESHOLDS = {
    'primary': {
        'Batter': dict(high=80, matches=8, ordinary=160),
        'Bowler': dict(high=100, matches=8, ordinary=200, high_clean=80, ordinary_clean=160),
    },
    'relaxed': {
        'Batter': dict(high=60, matches=6, ordinary=120),
        'Bowler': dict(high=75, matches=6, ordinary=150, high_clean=60, ordinary_clean=120),
    },
    'strict': {
        'Batter': dict(high=100, matches=10, ordinary=200),
        'Bowler': dict(high=125, matches=10, ordinary=250, high_clean=100, ordinary_clean=200),
    },
}
SHRINKAGE = {'Batter': np.array([80, 80, 80, 80]), 'Bowler': np.array([100, 80, 100, 80])}
BOWLER_WICKET_TYPES = {'Caught', 'Bowled', 'LBW', 'Stump', 'Hit Wicket'}

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def mode_or_first(series):
    values = series.dropna()
    if values.empty:
        return pd.NA
    modes = values.mode()
    return modes.iloc[0] if not modes.empty else values.iloc[0]

def markdown_table(frame, float_digits=4):
    columns = [str(c) for c in frame.columns]
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for row in frame.itertuples(index=False, name=None):
        rendered = []
        for value in row:
            if isinstance(value, (float, np.floating)):
                rendered.append(f'{value:.{float_digits}f}')
            else:
                rendered.append(str(value))
        lines.append('| ' + ' | '.join(rendered) + ' |')
    return '\n'.join(lines)

deliveries = pd.read_csv(INPUT_PATH, low_memory=False)
phase4_bat_format = pd.read_csv(PHASE4_DIR / 'phase4_batter_format_metrics.csv')
phase4_bowl_format = pd.read_csv(PHASE4_DIR / 'phase4_bowler_format_metrics.csv')
phase4_bat_combined = pd.read_csv(PHASE4_DIR / 'phase4_batter_combined_scores.csv')
phase4_bowl_combined = pd.read_csv(PHASE4_DIR / 'phase4_bowler_combined_scores.csv')
phase4_benchmarks = pd.read_csv(PHASE4_DIR / 'phase4_outcome_benchmarks.csv')
phase4_gates = pd.read_csv(PHASE4_DIR / 'phase4_validation_gates.csv')
phase4_manifest = pd.read_csv(PHASE4_DIR / 'phase4_output_manifest.csv')
require(phase4_gates['status'].eq('PASS').all(), 'Phase 4 contains failed validation gates')
require(deliveries['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all() if 'official_period' in deliveries else True, 'Unexpected period')

match_table = deliveries[['match_id', 'format_group']].drop_duplicates().sort_values(['format_group', 'match_id']).reset_index(drop=True)
match_table['match_idx'] = np.arange(len(match_table), dtype=np.int32)
match_lookup = match_table.set_index('match_id')['match_idx']
deliveries['match_idx'] = deliveries['match_id'].map(match_lookup).astype(np.int32)
print(f'Loaded {len(deliveries):,} deliveries across {len(match_table):,} matches.')

Loaded 188,330 deliveries across 789 matches.


## Reusable scoring engine

In [2]:
EXACT_KEYS = ['format_group', 'innings', 'overid', 'structural_pressure_category']
STAGE_KEYS = ['format_group', 'innings', 'source_over_stage', 'structural_pressure_category']
BROAD_KEYS = ['format_group', 'innings', 'structural_pressure_category']

def make_records(source):
    batting_mask = source['extras_type'].isna() | source['extras_type'].isin(['byes', 'legbyes'])
    bat = source.loc[batting_mask].copy()
    bat['batter_dismissal_available'] = ~(bat['is_wicket'] & bat['dismissed_player_id'].isna())
    bat['batter_dismissal'] = (bat['is_wicket'] & bat['dismissed_player_id'].notna() & bat['dismissed_player_id'].eq(bat['striker_id'])).astype(int)
    wicket = source.copy()
    wicket['bowler_wicket'] = (wicket['is_wicket'] & wicket['dismissal_type_normalized'].isin(BOWLER_WICKET_TYPES)).astype(int)
    run = source.loc[source['extras_type'].isna()].copy()
    common = ['match_idx', 'match_id', 'format_group', 'innings', 'overid', 'source_over_stage', 'structural_pressure_category', 'structural_pressure_percentile']
    records = {
        'batter_runs': bat[common + ['striker_id', 'striker_name', 'runs']].rename(columns={'striker_id': 'player_id', 'striker_name': 'player_name', 'runs': 'outcome'}),
        'batter_dismissal': bat.loc[bat['batter_dismissal_available'], common + ['striker_id', 'striker_name', 'batter_dismissal']].rename(columns={'striker_id': 'player_id', 'striker_name': 'player_name', 'batter_dismissal': 'outcome'}),
        'bowler_wicket': wicket[common + ['bowler_id', 'bowler_name', 'bowler_wicket']].rename(columns={'bowler_id': 'player_id', 'bowler_name': 'player_name', 'bowler_wicket': 'outcome'}),
        'bowler_batter_runs': run[common + ['bowler_id', 'bowler_name', 'runs']].rename(columns={'bowler_id': 'player_id', 'bowler_name': 'player_name', 'runs': 'outcome'}),
    }
    for frame in records.values():
        frame['x'] = frame['structural_pressure_percentile'].astype(float)
        frame['is_high'] = frame['structural_pressure_category'].eq('High')
    return records

def build_pf_table(records, base_metric, role):
    frame = records[base_metric]
    pf = frame.groupby(['player_id', 'format_group'], sort=True, observed=True).agg(player_name=('player_name', mode_or_first)).reset_index()
    pf['pf_idx'] = np.arange(len(pf), dtype=np.int32)
    pf['role'] = role
    players = pf.groupby('player_id', sort=True).agg(player_name=('player_name', mode_or_first)).reset_index()
    players['player_idx'] = np.arange(len(players), dtype=np.int32)
    pf = pf.merge(players[['player_id', 'player_idx']], on='player_id', validate='many_to_one')
    return pf, players

def make_sparse(frame, row_col, value_col, n_rows, n_matches):
    return sparse.csr_matrix((frame[value_col].astype(float), (frame[row_col].astype(int), frame['match_idx'].astype(int))), shape=(n_rows, n_matches))

def benchmark_table_from_records(frame):
    result = frame[EXACT_KEYS].drop_duplicates().sort_values(EXACT_KEYS).reset_index(drop=True)
    for label, keys in [('exact', EXACT_KEYS), ('stage', STAGE_KEYS), ('broad', BROAD_KEYS)]:
        stats = frame.groupby(keys, observed=True)['outcome'].agg(['size', 'mean']).reset_index().rename(columns={'size': f'{label}_support', 'mean': f'{label}_mean'})
        result = result.merge(stats, on=keys if label == 'exact' else [k for k in keys if k in result.columns], how='left', validate='many_to_one') if label == 'exact' else result
    stage_stats = frame.groupby(STAGE_KEYS, observed=True)['outcome'].agg(['size', 'mean']).reset_index().rename(columns={'size': 'stage_support', 'mean': 'stage_mean'})
    broad_stats = frame.groupby(BROAD_KEYS, observed=True)['outcome'].agg(['size', 'mean']).reset_index().rename(columns={'size': 'broad_support', 'mean': 'broad_mean'})
    stage_map = frame[EXACT_KEYS + ['source_over_stage']].drop_duplicates()
    result = result.merge(stage_map, on=EXACT_KEYS, validate='one_to_one').merge(stage_stats, on=STAGE_KEYS, validate='many_to_one').merge(broad_stats, on=BROAD_KEYS, validate='many_to_one')
    result['expected'] = np.where(result['exact_support'].ge(MIN_SUPPORT), result['exact_mean'], np.where(result['stage_support'].ge(MIN_SUPPORT), result['stage_mean'], result['broad_mean']))
    result['support'] = np.where(result['exact_support'].ge(MIN_SUPPORT), result['exact_support'], np.where(result['stage_support'].ge(MIN_SUPPORT), result['stage_support'], result['broad_support']))
    require(result['expected'].notna().all() and result['support'].ge(MIN_SUPPORT).all(), 'Full-sample benchmark fallback failed')
    return result

def build_metric_model(frame, metric_name, direction, pf, fixed_benchmark=None):
    n_matches = len(match_table)
    pf_lookup = pf.set_index(['player_id', 'format_group'])['pf_idx']
    working = frame.copy()
    working['pf_idx'] = [pf_lookup.get((pid, fmt), -1) for pid, fmt in zip(working['player_id'], working['format_group'])]
    working = working.loc[working['pf_idx'].ge(0)].copy()
    exact = working[EXACT_KEYS].drop_duplicates().sort_values(EXACT_KEYS).reset_index(drop=True)
    exact['exact_idx'] = np.arange(len(exact), dtype=np.int32)
    stage = working[STAGE_KEYS].drop_duplicates().sort_values(STAGE_KEYS).reset_index(drop=True)
    stage['stage_idx'] = np.arange(len(stage), dtype=np.int32)
    broad = working[BROAD_KEYS].drop_duplicates().sort_values(BROAD_KEYS).reset_index(drop=True)
    broad['broad_idx'] = np.arange(len(broad), dtype=np.int32)
    working = working.merge(exact, on=EXACT_KEYS, validate='many_to_one').merge(stage, on=STAGE_KEYS, validate='many_to_one').merge(broad, on=BROAD_KEYS, validate='many_to_one')
    exact_map = working[['exact_idx', 'stage_idx', 'broad_idx']].drop_duplicates().sort_values('exact_idx')
    require(exact_map['exact_idx'].is_unique, f'Non-unique context map for {metric_name}')
    grouped = working.groupby(['pf_idx', 'exact_idx', 'match_idx'], observed=True).agg(
        n=('outcome', 'size'), outcome=('outcome', 'sum'), x=('x', 'sum'),
        x2=('x', lambda v: np.square(v).sum()), xout=('outcome', lambda v: 0.0),
    ).reset_index()
    xout = working.assign(_xout=working['x'] * working['outcome']).groupby(['pf_idx', 'exact_idx', 'match_idx'], observed=True)['_xout'].sum().reset_index()
    grouped = grouped.drop(columns='xout').merge(xout, on=['pf_idx', 'exact_idx', 'match_idx'], validate='one_to_one').rename(columns={'_xout': 'xout'})
    pc = grouped[['pf_idx', 'exact_idx']].drop_duplicates().sort_values(['pf_idx', 'exact_idx']).reset_index(drop=True)
    pc['pc_idx'] = np.arange(len(pc), dtype=np.int32)
    grouped = grouped.merge(pc, on=['pf_idx', 'exact_idx'], validate='many_to_one')
    n_pc = len(pc)
    mats = {col: make_sparse(grouped, 'pc_idx', col, n_pc, n_matches) for col in ['n', 'outcome', 'x', 'x2', 'xout']}
    membership = sparse.csr_matrix((np.ones(n_pc), (pc['pf_idx'], pc['pc_idx'])), shape=(len(pf), n_pc))
    high_context = exact['structural_pressure_category'].eq('High').to_numpy()
    high_pc = high_context[pc['exact_idx'].to_numpy()]
    high_match_pairs = working.loc[working['is_high'], ['pf_idx', 'match_idx']].drop_duplicates()
    high_match_matrix = sparse.csr_matrix((np.ones(len(high_match_pairs)), (high_match_pairs['pf_idx'], high_match_pairs['match_idx'])), shape=(len(pf), n_matches))
    context_mats = {}
    for level, idx, size in [('exact', 'exact_idx', len(exact)), ('stage', 'stage_idx', len(stage)), ('broad', 'broad_idx', len(broad))]:
        ctx = working.groupby([idx, 'match_idx'], observed=True)['outcome'].agg(['size', 'sum']).reset_index()
        context_mats[f'{level}_n'] = make_sparse(ctx.rename(columns={idx: 'row_idx', 'size': 'value'}), 'row_idx', 'value', size, n_matches)
        context_mats[f'{level}_out'] = make_sparse(ctx.rename(columns={idx: 'row_idx', 'sum': 'value'}), 'row_idx', 'value', size, n_matches)
    if fixed_benchmark is None:
        fixed = benchmark_table_from_records(working)[EXACT_KEYS + ['expected']]
    else:
        fixed = fixed_benchmark.loc[fixed_benchmark['metric'].eq(metric_name), EXACT_KEYS + ['expected_outcome']].rename(columns={'expected_outcome': 'expected'})
    fixed = exact.merge(fixed, on=EXACT_KEYS, how='left', validate='one_to_one')
    require(fixed['expected'].notna().all(), f'Missing fixed benchmark for {metric_name}')
    return {
        'metric': metric_name, 'direction': float(direction), 'pf': pf, 'exact': exact,
        'pc_pf': pc['pf_idx'].to_numpy(), 'pc_exact': pc['exact_idx'].to_numpy(),
        'high_pc': high_pc, 'membership': membership, 'high_match_matrix': high_match_matrix,
        'mats': mats, 'context_mats': context_mats,
        'exact_to_stage': exact_map['stage_idx'].to_numpy(), 'exact_to_broad': exact_map['broad_idx'].to_numpy(),
        'fixed_expected': fixed['expected'].to_numpy(),
    }

def prepare_models(source, fixed_benchmark=None):
    records = make_records(source)
    bat_pf, bat_players = build_pf_table(records, 'batter_runs', 'Batter')
    bowl_pf, bowl_players = build_pf_table(records, 'bowler_wicket', 'Bowler')
    models = {
        'batter_runs': build_metric_model(records['batter_runs'], 'batter_runs', 1, bat_pf, fixed_benchmark),
        'batter_dismissal': build_metric_model(records['batter_dismissal'], 'batter_dismissal', -1, bat_pf, fixed_benchmark),
        'bowler_wicket': build_metric_model(records['bowler_wicket'], 'bowler_wicket', 1, bowl_pf, fixed_benchmark),
        'bowler_batter_runs': build_metric_model(records['bowler_batter_runs'], 'bowler_batter_runs', -1, bowl_pf, fixed_benchmark),
    }
    return models, {'Batter': bat_pf, 'Bowler': bowl_pf}, {'Batter': bat_players, 'Bowler': bowl_players}

models, pf_tables, player_tables = prepare_models(deliveries, phase4_benchmarks)
print('Prepared sparse sufficient-statistic models.')

Prepared sparse sufficient-statistic models.


In [3]:
def expected_for_weights(model, weights, mode):
    if mode == 'fixed':
        return np.repeat(model['fixed_expected'][:, None], weights.shape[1], axis=1)
    cm = model['context_mats']
    exact_n, exact_out = cm['exact_n'] @ weights, cm['exact_out'] @ weights
    stage_n, stage_out = cm['stage_n'] @ weights, cm['stage_out'] @ weights
    broad_n, broad_out = cm['broad_n'] @ weights, cm['broad_out'] @ weights
    exact_mean = np.divide(exact_out, exact_n, out=np.full_like(exact_out, np.nan), where=exact_n > 0)
    stage_mean = np.divide(stage_out, stage_n, out=np.full_like(stage_out, np.nan), where=stage_n > 0)
    broad_mean = np.divide(broad_out, broad_n, out=np.full_like(broad_out, np.nan), where=broad_n > 0)
    stage_idx = model['exact_to_stage']
    broad_idx = model['exact_to_broad']
    expected = np.where(exact_n >= MIN_SUPPORT, exact_mean, np.where(stage_n[stage_idx] >= MIN_SUPPORT, stage_mean[stage_idx], broad_mean[broad_idx]))
    require(np.isfinite(expected).all(), f"Nested benchmark failure: {model['metric']}")
    return expected

def metric_statistics(model, weights, mode):
    mats = model['mats']
    n = mats['n'] @ weights
    out = mats['outcome'] @ weights
    x = mats['x'] @ weights
    x2 = mats['x2'] @ weights
    xout = mats['xout'] @ weights
    expected = expected_for_weights(model, weights, mode)
    expected_pc = expected[model['pc_exact']]
    direction = model['direction']
    y = direction * (out - expected_pc * n)
    xy = direction * (xout - expected_pc * x)
    membership = model['membership']
    all_n, all_x, all_x2 = membership @ n, membership @ x, membership @ x2
    all_y, all_xy = membership @ y, membership @ xy
    high = model['high_pc'][:, None].astype(float)
    high_n = membership @ (n * high)
    high_y = membership @ (y * high)
    ordinary_n = all_n - high_n
    high_raw = np.divide(100 * high_y, high_n, out=np.full_like(high_y, np.nan), where=high_n > 0)
    denominator = all_x2 - np.divide(np.square(all_x), all_n, out=np.zeros_like(all_x), where=all_n > 0)
    numerator = all_xy - np.divide(all_x * all_y, all_n, out=np.zeros_like(all_y), where=all_n > 0)
    slope = np.divide(100 * numerator, denominator, out=np.full_like(numerator, np.nan), where=denominator > 0)
    high_matches = model['high_match_matrix'] @ weights
    return {'high_raw': high_raw, 'slope_raw': slope, 'high': high_n, 'ordinary': ordinary_n, 'high_matches': high_matches}

def assemble_raw(models, weights, mode):
    bat_run = metric_statistics(models['batter_runs'], weights, mode)
    bat_surv = metric_statistics(models['batter_dismissal'], weights, mode)
    bowl_wicket = metric_statistics(models['bowler_wicket'], weights, mode)
    bowl_run = metric_statistics(models['bowler_batter_runs'], weights, mode)
    return {
        'Batter': {
            'metrics': np.stack([bat_run['high_raw'], bat_surv['high_raw'], bat_run['slope_raw'], bat_surv['slope_raw']], axis=2),
            'high': bat_run['high'], 'ordinary': bat_run['ordinary'], 'matches': bat_run['high_matches'],
            'high_clean': None, 'ordinary_clean': None,
        },
        'Bowler': {
            'metrics': np.stack([bowl_wicket['high_raw'], bowl_run['high_raw'], bowl_wicket['slope_raw'], bowl_run['slope_raw']], axis=2),
            'high': bowl_wicket['high'], 'ordinary': bowl_wicket['ordinary'], 'matches': bowl_wicket['high_matches'],
            'high_clean': bowl_run['high'], 'ordinary_clean': bowl_run['ordinary'],
        },
    }

def eligibility_mask(role, raw, threshold):
    reliable = raw['high'] >= threshold['high']
    reliable &= raw['matches'] >= threshold['matches']
    reliable &= raw['ordinary'] >= threshold['ordinary']
    if role == 'Bowler':
        reliable &= raw['high_clean'] >= threshold['high_clean']
        reliable &= raw['ordinary_clean'] >= threshold['ordinary_clean']
    return reliable

def score_from_raw(raw_by_role, weights_by_role=PRIMARY_WEIGHTS, thresholds=THRESHOLDS['primary'], combination='equal', replicate_offset=0, mode_name='full', include_format=False):
    combined_rows, format_rows = [], []
    for role in ['Batter', 'Bowler']:
        pf = pf_tables[role]
        players = player_tables[role]
        raw = raw_by_role[role]
        reliable = eligibility_mask(role, raw, thresholds[role])
        metric_values = raw['metrics'].copy()
        evidence = np.stack([raw['high'], raw['high_clean'] if role == 'Bowler' else raw['high'], raw['high'], raw['high_clean'] if role == 'Bowler' else raw['high']], axis=2)
        shrink = SHRINKAGE[role][None, None, :]
        metrics = metric_values * evidence[:, :, None, 0] if False else metric_values
        metrics = metric_values * evidence / (evidence + shrink)
        n_reps = metrics.shape[1]
        ratings = np.full((len(pf), n_reps), np.nan)
        z_all = np.full_like(metrics, np.nan)
        for rep in range(n_reps):
            for fmt in FORMATS:
                pool = reliable[:, rep] & pf['format_group'].eq(fmt).to_numpy()
                require(pool.sum() >= 10, f'Too few reliable {role}s in {fmt}, replicate {replicate_offset + rep}')
                for metric_idx in range(4):
                    values = metrics[pool, rep, metric_idx]
                    require(np.isfinite(values).all(), f'Missing reliable {role} metric')
                    lower, upper = np.quantile(values, [0.025, 0.975])
                    wins = np.clip(values, lower, upper)
                    mean, sd = wins.mean(), wins.std(ddof=0)
                    require(sd > 0, f'Zero normalization SD for {role} {fmt}')
                    z_all[pool, rep, metric_idx] = (wins - mean) / sd
                ratings[pool, rep] = 50 + 10 * (z_all[pool, rep, :] @ weights_by_role[role])
            if include_format:
                for pf_idx, pf_row in pf.iterrows():
                    format_rows.append({
                        'bootstrap_mode': mode_name, 'replicate': replicate_offset + rep, 'role': role,
                        'player_id': int(pf_row.player_id), 'player_name': pf_row.player_name, 'format_group': pf_row.format_group,
                        'reliable': bool(reliable[pf_idx, rep]), 'format_rating': ratings[pf_idx, rep],
                        'high_records': raw['high'][pf_idx, rep], 'ordinary_records': raw['ordinary'][pf_idx, rep],
                        'high_matches': raw['matches'][pf_idx, rep],
                        'high_clean_records': raw['high_clean'][pf_idx, rep] if role == 'Bowler' else np.nan,
                        'ordinary_clean_records': raw['ordinary_clean'][pf_idx, rep] if role == 'Bowler' else np.nan,
                        'metric_1': metrics[pf_idx, rep, 0], 'metric_2': metrics[pf_idx, rep, 1],
                        'metric_3': metrics[pf_idx, rep, 2], 'metric_4': metrics[pf_idx, rep, 3],
                    })
            ipl = np.full(len(players), np.nan); t20i = np.full(len(players), np.nan)
            ipl_ev = np.zeros(len(players)); t20i_ev = np.zeros(len(players))
            for pf_idx, pf_row in pf.iterrows():
                pidx = int(pf_row.player_idx)
                if pf_row.format_group == 'IPL':
                    ipl[pidx] = ratings[pf_idx, rep]; ipl_ev[pidx] = raw['high'][pf_idx, rep] + raw['ordinary'][pf_idx, rep]
                else:
                    t20i[pidx] = ratings[pf_idx, rep]; t20i_ev[pidx] = raw['high'][pf_idx, rep] + raw['ordinary'][pf_idx, rep]
            valid_ipl, valid_t20i = np.isfinite(ipl), np.isfinite(t20i)
            both = valid_ipl & valid_t20i
            combined = np.where(valid_ipl, ipl, t20i)
            if combination == 'equal':
                combined[both] = (ipl[both] + t20i[both]) / 2
            elif combination == 'evidence_weighted':
                combined[both] = (ipl[both] * ipl_ev[both] + t20i[both] * t20i_ev[both]) / (ipl_ev[both] + t20i_ev[both])
            elif combination == 'conservative_min':
                combined[both] = np.minimum(ipl[both], t20i[both])
            else:
                raise ValueError(f'Unknown format combination {combination}')
            valid = np.isfinite(combined)
            ids = players['player_id'].to_numpy()
            order = np.lexsort((ids[valid], -combined[valid]))
            valid_idx = np.flatnonzero(valid)[order]
            for rank, pidx in enumerate(valid_idx, start=1):
                combined_rows.append({
                    'bootstrap_mode': mode_name, 'replicate': replicate_offset + rep, 'role': role,
                    'player_id': int(players.iloc[pidx].player_id), 'player_name': players.iloc[pidx].player_name,
                    'qualifying_formats': 'IPL+T20I' if both[pidx] else ('IPL' if valid_ipl[pidx] else 'T20I'),
                    'qualifying_format_count': 2 if both[pidx] else 1,
                    'ipl_rating': ipl[pidx], 't20i_rating': t20i[pidx], 'combined_rating': combined[pidx],
                    'rank': rank, 'top5': rank <= 5, 'top10': rank <= 10,
                })
    return pd.DataFrame(combined_rows), pd.DataFrame(format_rows)

ones = np.ones((len(match_table), 1), dtype=np.int16)
official_raw = assemble_raw(models, ones, 'fixed')
official_results, official_format = score_from_raw(official_raw, replicate_offset=0, mode_name='official', include_format=True)
print('Scoring engine complete.')

Scoring engine complete.


## Phase 4 reproduction and official rankings

In [4]:
phase4_combined = pd.concat([phase4_bat_combined, phase4_bowl_combined], ignore_index=True)
reproduction = official_results.merge(phase4_combined[['role', 'player_id', 'combined_rating', 'ipl_rating', 't20i_rating']], on=['role', 'player_id'], how='outer', suffixes=('_recomputed', '_phase4'), indicator=True)
require(reproduction['_merge'].eq('both').all(), 'Phase 4 candidate population was not reproduced')
for column in ['combined_rating', 'ipl_rating', 't20i_rating']:
    a, b = reproduction[f'{column}_recomputed'], reproduction[f'{column}_phase4']
    require(np.allclose(a, b, rtol=0, atol=1e-8, equal_nan=True), f'Phase 4 {column} was not reproduced')
official_rankings = official_results.copy()
official_rankings['official_top10'] = official_rankings['rank'].le(10)
official_rankings['gap_to_rank10'] = np.nan
official_rankings['gap_to_rank11'] = np.nan
for role, group in official_rankings.groupby('role'):
    rank10 = group.loc[group['rank'].eq(10), 'combined_rating'].iloc[0]
    rank11 = group.loc[group['rank'].eq(11), 'combined_rating'].iloc[0]
    mask = official_rankings['role'].eq(role)
    official_rankings.loc[mask, 'gap_to_rank10'] = official_rankings.loc[mask, 'combined_rating'] - rank10
    official_rankings.loc[mask, 'gap_to_rank11'] = official_rankings.loc[mask, 'combined_rating'] - rank11
official_rankings['phase5_model_version'] = MODEL_VERSION
print(official_rankings.loc[official_rankings['official_top10'], ['role', 'rank', 'player_id', 'player_name', 'qualifying_formats', 'combined_rating']].to_string(index=False))

  role  rank  player_id        player_name qualifying_formats  combined_rating
Batter     1       3125    Nicholas Pooran                IPL        62.212052
Batter     2       3323      Jimmy Neesham               T20I        61.845620
Batter     3       3330      Hardik Pandya           IPL+T20I        60.482993
Batter     4        813        Jos Buttler                IPL        60.390413
Batter     5       4773     Jaker Ali Anik               T20I        60.123330
Batter     6       6726   Romario Shepherd               T20I        59.276990
Batter     7       1699   Suryakumar Yadav           IPL+T20I        59.215134
Batter     8       3056     Iftikhar Ahmed               T20I        58.774199
Batter     9       3647       Shreyas Iyer                IPL        58.419471
Batter    10       4708        Rinku Singh           IPL+T20I        57.429273
Bowler     1      34485     Sufiyan Muqeem               T20I        69.666764
Bowler     2       3644      Kuldeep Yadav          

## Match-level bootstrap

In [5]:
def bootstrap_weights(replicates, seed):
    rng = np.random.default_rng(seed)
    weights = np.zeros((len(match_table), replicates), dtype=np.int16)
    for fmt in FORMATS:
        indices = match_table.index[match_table['format_group'].eq(fmt)].to_numpy()
        draws = rng.integers(0, len(indices), size=(replicates, len(indices)))
        for rep in range(replicates):
            weights[indices, rep] = np.bincount(draws[rep], minlength=len(indices))
    return weights

def run_bootstrap(replicates, seed, mode, batch_size):
    weights = bootstrap_weights(replicates, seed)
    rows = []
    for start in range(0, replicates, batch_size):
        stop = min(start + batch_size, replicates)
        batch_weights = weights[:, start:stop]
        raw = assemble_raw(models, batch_weights, mode)
        combined, _ = score_from_raw(raw, replicate_offset=start + 1, mode_name=mode)
        rows.append(combined)
        if stop % 500 == 0 or stop == replicates:
            print(f'{mode}: completed {stop:,}/{replicates:,} replicates', flush=True)
    return pd.concat(rows, ignore_index=True)

fixed_bootstrap = run_bootstrap(FIXED_REPLICATES, FIXED_SEED, 'fixed', FIXED_BATCH)
nested_bootstrap = run_bootstrap(NESTED_REPLICATES, NESTED_SEED, 'nested', NESTED_BATCH)
bootstrap_results = pd.concat([fixed_bootstrap, nested_bootstrap], ignore_index=True)
print(f'Bootstrap rows: {len(bootstrap_results):,}')

fixed: completed 500/5,000 replicates
fixed: completed 1,000/5,000 replicates
fixed: completed 1,500/5,000 replicates
fixed: completed 2,000/5,000 replicates
fixed: completed 2,500/5,000 replicates
fixed: completed 3,000/5,000 replicates
fixed: completed 3,500/5,000 replicates
fixed: completed 4,000/5,000 replicates
fixed: completed 4,500/5,000 replicates
fixed: completed 5,000/5,000 replicates
nested: completed 500/1,000 replicates
nested: completed 1,000/1,000 replicates
Bootstrap rows: 1,082,184


In [6]:
def bootstrap_summary(results):
    rows = []
    reps_by_mode = {'fixed': FIXED_REPLICATES, 'nested': NESTED_REPLICATES}
    for (mode, role, player_id, player_name), group in results.groupby(['bootstrap_mode', 'role', 'player_id', 'player_name'], observed=True):
        total = reps_by_mode[mode]
        top10_p = group['top10'].sum() / total
        rows.append({
            'bootstrap_mode': mode, 'role': role, 'player_id': int(player_id), 'player_name': player_name,
            'total_replicates': total, 'ranked_replicates': group['replicate'].nunique(),
            'ranked_frequency': group['replicate'].nunique() / total,
            'rating_mean': group['combined_rating'].mean(), 'rating_median': group['combined_rating'].median(),
            'rating_p025': group['combined_rating'].quantile(.025), 'rating_p975': group['combined_rating'].quantile(.975),
            'rank_mean': group['rank'].mean(), 'rank_median': group['rank'].median(),
            'rank_p025': group['rank'].quantile(.025), 'rank_p975': group['rank'].quantile(.975),
            'top5_frequency': group['top5'].sum() / total, 'top10_frequency': top10_p,
            'top10_monte_carlo_se': math.sqrt(top10_p * (1 - top10_p) / total),
        })
    return pd.DataFrame(rows)

bootstrap_summary_table = bootstrap_summary(bootstrap_results)
print(bootstrap_summary_table.sort_values(['role', 'bootstrap_mode', 'top10_frequency'], ascending=[True, True, False]).groupby(['role', 'bootstrap_mode']).head(12).to_string(index=False))

bootstrap_mode   role  player_id        player_name  total_replicates  ranked_replicates  ranked_frequency  rating_mean  rating_median  rating_p025  rating_p975  rank_mean  rank_median  rank_p025  rank_p975  top5_frequency  top10_frequency  top10_monte_carlo_se
         fixed Batter       3330      Hardik Pandya              5000               4998            0.9996    59.019279      59.158506    51.759046    65.241006   9.804722          8.0        1.0     32.000          0.3636           0.6564              0.006716
         fixed Batter       3125    Nicholas Pooran              5000               4999            0.9998    58.374847      58.589039    48.678864    66.575932  12.294459          9.0        1.0     44.000          0.3426           0.5704              0.007001
         fixed Batter       3323      Jimmy Neesham              5000               4687            0.9374    58.098045      58.561035    46.210850    67.361043  14.086623          9.0        1.0     52.000        

## Self-influence and deterministic robustness scenarios

In [7]:
def lopo_metric_statistics(model):
    n_pf = len(model['pf'])
    result_high = np.full((n_pf, 1), np.nan)
    result_slope = np.full((n_pf, 1), np.nan)
    one = np.ones((len(match_table), 1))
    cm = model['context_mats']
    global_exact_n = np.asarray(cm['exact_n'] @ one).ravel(); global_exact_out = np.asarray(cm['exact_out'] @ one).ravel()
    global_stage_n = np.asarray(cm['stage_n'] @ one).ravel(); global_stage_out = np.asarray(cm['stage_out'] @ one).ravel()
    global_broad_n = np.asarray(cm['broad_n'] @ one).ravel(); global_broad_out = np.asarray(cm['broad_out'] @ one).ravel()
    pc_n = np.asarray(model['mats']['n'] @ one).ravel(); pc_out = np.asarray(model['mats']['outcome'] @ one).ravel()
    pc_x = np.asarray(model['mats']['x'] @ one).ravel(); pc_x2 = np.asarray(model['mats']['x2'] @ one).ravel(); pc_xout = np.asarray(model['mats']['xout'] @ one).ravel()
    for pf_idx in range(n_pf):
        rows = np.flatnonzero(model['pc_pf'] == pf_idx)
        if not len(rows):
            continue
        ex = model['pc_exact'][rows]
        p_ex_n = np.bincount(ex, weights=pc_n[rows], minlength=len(global_exact_n)); p_ex_out = np.bincount(ex, weights=pc_out[rows], minlength=len(global_exact_n))
        st = model['exact_to_stage'][ex]; br = model['exact_to_broad'][ex]
        p_st_n = np.bincount(st, weights=pc_n[rows], minlength=len(global_stage_n)); p_st_out = np.bincount(st, weights=pc_out[rows], minlength=len(global_stage_n))
        p_br_n = np.bincount(br, weights=pc_n[rows], minlength=len(global_broad_n)); p_br_out = np.bincount(br, weights=pc_out[rows], minlength=len(global_broad_n))
        en, eo = global_exact_n - p_ex_n, global_exact_out - p_ex_out
        sn, so = global_stage_n - p_st_n, global_stage_out - p_st_out
        bn, bo = global_broad_n - p_br_n, global_broad_out - p_br_out
        em = np.divide(eo, en, out=np.full_like(eo, np.nan), where=en > 0); sm = np.divide(so, sn, out=np.full_like(so, np.nan), where=sn > 0); bm = np.divide(bo, bn, out=np.full_like(bo, np.nan), where=bn > 0)
        expected = np.where(en >= MIN_SUPPORT, em, np.where(sn[model['exact_to_stage']] >= MIN_SUPPORT, sm[model['exact_to_stage']], bm[model['exact_to_broad']]))
        require(np.isfinite(expected[ex]).all(), f"LOPO fallback failed for {model['metric']} pf {pf_idx}")
        exp = expected[ex]; direction = model['direction']
        y = direction * (pc_out[rows] - exp * pc_n[rows]); xy = direction * (pc_xout[rows] - exp * pc_x[rows])
        high = model['high_pc'][rows]
        if pc_n[rows][high].sum() > 0:
            result_high[pf_idx, 0] = 100 * y[high].sum() / pc_n[rows][high].sum()
        n, sx, sx2, sy, sxy = pc_n[rows].sum(), pc_x[rows].sum(), pc_x2[rows].sum(), y.sum(), xy.sum()
        den = sx2 - sx * sx / n if n else 0
        if den > 0:
            result_slope[pf_idx, 0] = 100 * (sxy - sx * sy / n) / den
    return result_high, result_slope

lopo_bat_run = lopo_metric_statistics(models['batter_runs']); lopo_bat_surv = lopo_metric_statistics(models['batter_dismissal'])
lopo_bowl_wicket = lopo_metric_statistics(models['bowler_wicket']); lopo_bowl_run = lopo_metric_statistics(models['bowler_batter_runs'])
lopo_raw = {
    'Batter': {**official_raw['Batter'], 'metrics': np.stack([lopo_bat_run[0], lopo_bat_surv[0], lopo_bat_run[1], lopo_bat_surv[1]], axis=2)},
    'Bowler': {**official_raw['Bowler'], 'metrics': np.stack([lopo_bowl_wicket[0], lopo_bowl_run[0], lopo_bowl_wicket[1], lopo_bowl_run[1]], axis=2)},
}
lopo_rankings, _ = score_from_raw(lopo_raw, mode_name='lopo')
self_influence = official_rankings[['role', 'player_id', 'player_name', 'rank', 'combined_rating', 'official_top10']].merge(
    lopo_rankings[['role', 'player_id', 'rank', 'combined_rating']].rename(columns={'rank': 'lopo_rank', 'combined_rating': 'lopo_rating'}),
    on=['role', 'player_id'], how='left', validate='one_to_one'
)
self_influence['rating_change'] = self_influence['lopo_rating'] - self_influence['combined_rating']
self_influence['rank_change'] = self_influence['lopo_rank'] - self_influence['rank']
self_influence['top10_status_changed'] = self_influence['official_top10'].ne(self_influence['lopo_rank'].le(10))
print('LOPO self-influence complete.')

LOPO self-influence complete.


In [8]:
scenario_frames = [official_rankings.assign(scenario='primary')]
for name, scenario_weights in WEIGHT_SCENARIOS.items():
    if name == 'primary':
        continue
    ranked, _ = score_from_raw(official_raw, weights_by_role=scenario_weights, mode_name=name)
    scenario_frames.append(ranked.assign(scenario=f'weights__{name}'))
for name in ['relaxed', 'strict']:
    ranked, _ = score_from_raw(official_raw, thresholds=THRESHOLDS[name], mode_name=name)
    scenario_frames.append(ranked.assign(scenario=f'eligibility__{name}'))
for combination in ['evidence_weighted', 'conservative_min']:
    ranked, _ = score_from_raw(official_raw, combination=combination, mode_name=combination)
    scenario_frames.append(ranked.assign(scenario=f'format__{combination}'))

pressure_variants = {
    'pressure__equal_w6': (1/3, 1/3, 1/3, 6),
    'pressure__equal_w18': (1/3, 1/3, 1/3, 18),
    'pressure__wicket_heavy': (0.50, 0.25, 0.25, 12),
    'pressure__shock_heavy': (0.25, 0.50, 0.25, 12),
    'pressure__stage_heavy': (0.25, 0.25, 0.50, 12),
}
for name, (w_wicket, w_shock, w_stage, window) in pressure_variants.items():
    variant = deliveries.copy()
    variant['structural_pressure_index'] = (
        w_wicket * variant['wicket_strain_component']
        + w_shock * variant[f'recent_wicket_shock_component_{window}']
        + w_stage * variant['stage_urgency_component']
    )
    variant['structural_pressure_percentile'] = variant.groupby(['format_group', 'innings'], observed=True)['structural_pressure_index'].rank(method='average', pct=True)
    variant['structural_pressure_category'] = np.select([variant['structural_pressure_percentile'].le(.40), variant['structural_pressure_percentile'].le(.80)], ['Low', 'Medium'], default='High')
    variant_models, variant_pf, variant_players = prepare_models(variant, fixed_benchmark=None)
    original_pf, original_players = pf_tables, player_tables
    pf_tables, player_tables = variant_pf, variant_players
    variant_raw = assemble_raw(variant_models, ones, 'nested')
    ranked, _ = score_from_raw(variant_raw, mode_name=name)
    pf_tables, player_tables = original_pf, original_players
    scenario_frames.append(ranked.assign(scenario=name))
    print(f'Completed {name}', flush=True)
scenario_rankings = pd.concat(scenario_frames, ignore_index=True)
require(scenario_rankings['scenario'].nunique() == 13, 'Unexpected scenario count')

primary_compare = scenario_rankings.loc[scenario_rankings['scenario'].eq('primary'), ['role', 'player_id', 'rank', 'player_name']].rename(columns={'rank': 'primary_rank'})
robustness_rows = []
for (scenario, role), group in scenario_rankings.groupby(['scenario', 'role'], observed=True):
    merged = primary_compare.loc[primary_compare['role'].eq(role)].merge(group[['player_id', 'rank']], on='player_id', how='inner')
    primary_top = set(primary_compare.loc[(primary_compare['role'].eq(role)) & (primary_compare['primary_rank'].le(10)), 'player_id'])
    scenario_top = set(group.loc[group['rank'].le(10), 'player_id'])
    union = primary_top | scenario_top
    name_lookup = dict(zip(group.player_id, group.player_name))
    robustness_rows.append({
        'scenario': scenario, 'role': role, 'common_players': len(merged),
        'spearman_vs_primary': spearmanr(merged['primary_rank'], merged['rank']).statistic if len(merged) > 1 else np.nan,
        'kendall_vs_primary': kendalltau(merged['primary_rank'], merged['rank']).statistic if len(merged) > 1 else np.nan,
        'top10_overlap': len(primary_top & scenario_top), 'top10_jaccard': len(primary_top & scenario_top) / len(union) if union else 1.0,
        'max_absolute_rank_movement': (merged['rank'] - merged['primary_rank']).abs().max(),
        'top10_entries': ', '.join(name_lookup.get(pid, str(pid)) for pid in sorted(scenario_top - primary_top)),
        'top10_exits': ', '.join(dict(zip(primary_compare.loc[primary_compare['role'].eq(role), 'player_id'], primary_compare.loc[primary_compare['role'].eq(role), 'player_name'])).get(pid, str(pid)) for pid in sorted(primary_top - scenario_top)),
    })
robustness_summary = pd.DataFrame(robustness_rows)
print('Deterministic robustness scenarios complete.')

Completed pressure__equal_w6
Completed pressure__equal_w18
Completed pressure__wicket_heavy
Completed pressure__shock_heavy
Completed pressure__stage_heavy
Deterministic robustness scenarios complete.


## Boundary classification, validation and export

In [9]:
bootstrap_wide = bootstrap_summary_table.pivot_table(index=['role', 'player_id', 'player_name'], columns='bootstrap_mode', values=['ranked_frequency', 'top10_frequency'], aggfunc='first').reset_index()
bootstrap_wide.columns = ['_'.join([str(x) for x in col if str(x)]) if isinstance(col, tuple) else col for col in bootstrap_wide.columns]
scenario_top_counts = scenario_rankings.assign(_top10=scenario_rankings['rank'].le(10)).groupby(['role', 'player_id'], observed=True)['_top10'].sum().rename('scenario_top10_count').reset_index()
classification = official_rankings.merge(bootstrap_wide, on=['role', 'player_id', 'player_name'], how='left').merge(scenario_top_counts, on=['role', 'player_id'], how='left')
for column in ['ranked_frequency_fixed', 'ranked_frequency_nested', 'top10_frequency_fixed', 'top10_frequency_nested', 'scenario_top10_count']:
    classification[column] = classification[column].fillna(0)
classification['robustness_class'] = 'OUTSIDE'
core = classification['official_top10'] & classification['top10_frequency_fixed'].ge(.75) & classification['top10_frequency_nested'].ge(.75)
classification.loc[core, 'robustness_class'] = 'CORE_TOP10'
classification.loc[classification['official_top10'] & ~core, 'robustness_class'] = 'BOUNDARY_TOP10'
challenger = ~classification['official_top10'] & (classification['rank'].between(11, 15) | classification['top10_frequency_fixed'].ge(.25) | classification['top10_frequency_nested'].ge(.25) | classification['scenario_top10_count'].gt(0))
classification.loc[challenger, 'robustness_class'] = 'CHALLENGER'

phase4_all_format = pd.concat([phase4_bat_format, phase4_bowl_format], ignore_index=True)
provisional = phase4_all_format.loc[phase4_all_format['eligibility'].eq('Provisional'), ['role', 'player_id', 'player_name']].drop_duplicates()
bootstrap_entrants = bootstrap_summary_table.loc[bootstrap_summary_table['top10_frequency'].gt(0), ['role', 'player_id', 'player_name']].drop_duplicates()
scenario_entrants = scenario_rankings.loc[scenario_rankings['rank'].le(10), ['role', 'player_id', 'player_name']].drop_duplicates()
provisional_appendix = provisional.merge(pd.concat([bootstrap_entrants, scenario_entrants]).drop_duplicates(), on=['role', 'player_id'], how='inner', suffixes=('_phase4', '_result'))
provisional_appendix['robustness_class'] = 'PROVISIONAL_CHALLENGER'
boundary_candidates = classification.loc[classification['robustness_class'].ne('OUTSIDE')].copy()
if len(provisional_appendix):
    provisional_export = provisional_appendix.rename(columns={'player_name_phase4': 'player_name'})[['role', 'player_id', 'player_name', 'robustness_class']]
    boundary_candidates = pd.concat([boundary_candidates, provisional_export], ignore_index=True, sort=False)

preupdate_invariance = {
    'Batter': sha256(PHASE4_DIR / 'phase4_batter_combined_scores.csv') == PREUPDATE_COMBINED_HASHES['Batter'],
    'Bowler': sha256(PHASE4_DIR / 'phase4_bowler_combined_scores.csv') == PREUPDATE_COMBINED_HASHES['Bowler'],
}
top10_per_replicate = bootstrap_results.groupby(['bootstrap_mode', 'role', 'replicate'], observed=True)['top10'].sum()
checks = [
    ('phase4_gates_pass', phase4_gates['status'].eq('PASS').all(), int(phase4_gates['status'].ne('PASS').sum()), '0 failed Phase 4 gates'),
    ('phase4_population_reproduced', reproduction['_merge'].eq('both').all(), int(reproduction['_merge'].ne('both').sum()), '0 population differences'),
    ('phase4_scores_reproduced', np.allclose(reproduction['combined_rating_recomputed'], reproduction['combined_rating_phase4'], rtol=0, atol=1e-8), float((reproduction['combined_rating_recomputed'] - reproduction['combined_rating_phase4']).abs().max()), 'max difference <= 1e-8'),
    ('official_top10_counts', official_rankings.groupby('role')['official_top10'].sum().eq(10).all(), official_rankings.groupby('role')['official_top10'].sum().to_dict(), '10 per role'),
    ('official_ids_unique', not official_rankings.duplicated(['role', 'player_id']).any(), int(official_rankings.duplicated(['role', 'player_id']).sum()), '0 duplicates'),
    ('fixed_replicates_complete', fixed_bootstrap['replicate'].nunique() == FIXED_REPLICATES, fixed_bootstrap['replicate'].nunique(), FIXED_REPLICATES),
    ('nested_replicates_complete', nested_bootstrap['replicate'].nunique() == NESTED_REPLICATES, nested_bootstrap['replicate'].nunique(), NESTED_REPLICATES),
    ('bootstrap_top10_reconciles', top10_per_replicate.eq(10).all(), int(top10_per_replicate.ne(10).sum()), '0 replicate-role groups not equal to 10'),
    ('bootstrap_frequency_bounds', bootstrap_summary_table[['ranked_frequency', 'top5_frequency', 'top10_frequency']].apply(lambda c: c.between(0, 1).all()).all(), 0, 'all frequencies in [0,1]'),
    ('bootstrap_quantiles_ordered', (bootstrap_summary_table['rating_p025'].le(bootstrap_summary_table['rating_median']) & bootstrap_summary_table['rating_median'].le(bootstrap_summary_table['rating_p975']) & bootstrap_summary_table['rank_p025'].le(bootstrap_summary_table['rank_median']) & bootstrap_summary_table['rank_median'].le(bootstrap_summary_table['rank_p975'])).all(), 0, 'all quantiles ordered'),
    ('scenario_count', scenario_rankings['scenario'].nunique() == 13, scenario_rankings['scenario'].nunique(), 13),
    ('scenario_top10_counts', scenario_rankings.groupby(['scenario', 'role'])['top10'].sum().eq(10).all(), int(scenario_rankings.groupby(['scenario', 'role'])['top10'].sum().ne(10).sum()), '10 per scenario and role'),
    ('self_influence_complete', self_influence['lopo_rating'].notna().all(), int(self_influence['lopo_rating'].isna().sum()), '0 missing LOPO ratings'),
    ('updated_dataset_invariance', all(preupdate_invariance.values()), preupdate_invariance, 'both combined-score hashes unchanged'),
    ('aneurin_alias_not_split', not official_rankings['player_name'].eq('AHT Donald').any() and not official_rankings.duplicated(['role', 'player_id']).any(), int(official_rankings['player_name'].eq('AHT Donald').sum()), '0 AHT Donald ranking rows'),
    ('no_recent_weight', phase4_combined['official_period'].eq('FULL_2022_2026_NO_RECENCY_WEIGHT').all(), phase4_combined['official_period'].unique().tolist(), 'full period only'),
]
validation = pd.DataFrame(checks, columns=['check_name', 'passed', 'observed', 'acceptance_rule'])
validation['status'] = np.where(validation['passed'], 'PASS', 'FAIL')
require(validation['passed'].all(), 'Phase 5 validation failed:\n' + validation.loc[~validation['passed']].to_string(index=False))
print(validation[['check_name', 'status']].to_string(index=False))

                  check_name status
           phase4_gates_pass   PASS
phase4_population_reproduced   PASS
    phase4_scores_reproduced   PASS
       official_top10_counts   PASS
         official_ids_unique   PASS
   fixed_replicates_complete   PASS
  nested_replicates_complete   PASS
  bootstrap_top10_reconciles   PASS
  bootstrap_frequency_bounds   PASS
 bootstrap_quantiles_ordered   PASS
              scenario_count   PASS
       scenario_top10_counts   PASS
     self_influence_complete   PASS
  updated_dataset_invariance   PASS
     aneurin_alias_not_split   PASS
            no_recent_weight   PASS


In [10]:
methodology = pd.DataFrame([
    ('official_period', 'FULL_2022_2026_NO_RECENCY_WEIGHT', 'No recency weighting'),
    ('ranking_rule', 'unrounded combined rating descending; player_id ascending tie-break', 'Exactly 10 per role'),
    ('fixed_bootstrap', f'{FIXED_REPLICATES} stratified match replicates; seed {FIXED_SEED}', 'Phase 3 labels and Phase 4 benchmarks fixed'),
    ('nested_bootstrap', f'{NESTED_REPLICATES} stratified match replicates; seed {NESTED_SEED}', 'Phase 4 outcome benchmarks rebuilt'),
    ('bootstrap_formats', 'IPL and T20I resampled separately', '357 IPL and 432 T20I draws per replicate'),
    ('normalization', 'recomputed within role-format reliable pool per replicate/scenario', '2.5/97.5 winsorization'),
    ('self_influence', 'leave evaluated player out of outcome benchmarks', 'Same support and fallback hierarchy'),
    ('core_top10', 'official top10 and >=75% inclusion in both bootstrap modes', 'Descriptive uncertainty class'),
    ('boundary_top10', 'official top10 but not core', 'No automatic replacement'),
    ('challenger', 'rank 11-15 or >=25% inclusion or any scenario top10', 'Review boundary case'),
], columns=['item', 'locked_rule', 'note'])
methodology['phase5_model_version'] = MODEL_VERSION

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifacts = {
    'phase5_official_rankings.csv': official_rankings,
    'phase5_bootstrap_player_results.csv': bootstrap_results,
    'phase5_bootstrap_summary.csv': bootstrap_summary_table,
    'phase5_scenario_rankings.csv': scenario_rankings,
    'phase5_robustness_summary.csv': robustness_summary,
    'phase5_self_influence.csv': self_influence,
    'phase5_boundary_and_provisional_candidates.csv': boundary_candidates,
    'phase5_validation_gates.csv': validation,
    'phase5_methodology_specification.csv': methodology,
}
for name, frame in artifacts.items():
    frame.to_csv(OUTPUT_DIR / name, index=False, float_format='%.10g')
manifest_rows = []
for name, frame in artifacts.items():
    path = OUTPUT_DIR / name
    manifest_rows.append({
        'artifact': name, 'rows': len(frame), 'columns': len(frame.columns), 'bytes': path.stat().st_size,
        'sha256': sha256(path), 'model_version': MODEL_VERSION, 'source_sha256': sha256(INPUT_PATH),
    })
manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / 'phase5_output_manifest.csv', index=False)

official_display = official_rankings.loc[official_rankings['official_top10'], ['role', 'rank', 'player_name', 'qualifying_formats', 'combined_rating']].copy()
class_display = classification.loc[classification['official_top10'], ['role', 'rank', 'player_name', 'robustness_class', 'top10_frequency_fixed', 'top10_frequency_nested']].copy()
report = f'''# Phase 5 — Rankings, Uncertainty and Robustness Report

## Decision

Official rankings use unrounded Phase 4 combined ratings over the complete eligible 2022–2026 period. No recent-form weight is used. Bootstrap and sensitivity results diagnose uncertainty; they do not silently replace the official score order.

## Official top 10

{markdown_table(official_display)}

## Bootstrap design

The analysis uses {FIXED_REPLICATES:,} fixed-benchmark and {NESTED_REPLICATES:,} nested-benchmark stratified match replicates. IPL and T20I matches are resampled separately. Eligibility, shrinkage, normalization, format scores and combined ratings are recalculated in each replicate.

## Top-10 uncertainty

{markdown_table(class_display)}

Core top-10 players have at least 75% inclusion in both bootstrap modes. Other official selections remain valid by the locked full-data ranking but are identified as boundary cases. Challengers are reported separately rather than substituted automatically.

## Robustness

The deterministic suite contains {scenario_rankings.scenario.nunique()} factor-at-a-time scenarios covering metric weights, eligibility thresholds, pressure definitions and format combination. Leave-one-player-out outcome benchmarks separately test benchmark self-influence.

## Validation

All {len(validation)} Phase 5 gates passed. The notebook reproduced the Phase 4 candidate population and combined ratings, reconciled exactly 10 top-10 selections per role in every bootstrap replicate and deterministic scenario, and verified that the updated dataset did not change the official combined-score hashes.

## Phase 6 handoff

Phase 6 should use the official rankings together with bootstrap inclusion, self-influence, scenario stability and boundary classifications when explaining each selection. It must distinguish full-data rank from uncertainty evidence and preserve provisional status for non-official candidates.
'''
REPORT_PATH.write_text(report, encoding='utf-8')
print(manifest.to_string(index=False))
print('PHASE5_COMPLETE')

                                      artifact    rows  columns    bytes                                                           sha256          model_version                                                    source_sha256
                  phase5_official_rankings.csv     185       17    26174 a53a521603b724d502c111629ab34d7b7194de6866d3f1bb51456ec752331abb ranking_uncertainty_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
           phase5_bootstrap_player_results.csv 1082184       13 92625486 cff8288916d7e7b694911870fb2eba29be0c3e8db39047f53336e6be322bb70e ranking_uncertainty_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
                  phase5_bootstrap_summary.csv     818       18   112205 4d94342eae87f18918d41e9f83a8009de6ba35392fa123e2eddc76c9aedc5bdb ranking_uncertainty_v1 d6706b02c9e3ac916432319ae3309cf1daedd9a39feecbb93b8a0bb2680f7798
                  phase5_scenario_rankings.csv    2384       18   295304 e207aa5ab7b417461535085